## Table extraction with OCR and layout model
Three tables are extracted <br>
Top table with voter information <br>
Bottom table with Votes for each party <br>
Key value pair with code of the polling Unit <br>
Each image/ Polling unit result is extract to one CSV <br>
All results for each table are written into one csv with PU-code and image path

In [ ]:
# Define textract key and secret
# access_key = 'XXX'
# secret = 'XXX'

In [2]:
from PIL import Image, ImageDraw, ImageOps, ImageEnhance, ImageFont
import boto3
from botocore.exceptions import NoCredentialsError, PartialCredentialsError
import pandas as pd
import cv2
import os
import numpy as np
import csv
import random
import glob
import os
from ultralytics import YOLO
from tqdm import tqdm
import torch
import argparse
import torch.nn.functional as F
import torch.nn as nn
from shapely.geometry import Polygon
import pandas as pd
from fuzzywuzzy import process, fuzz
import random
import re
pd.options.mode.chained_assignment = None
det_model = YOLO("../models/rotation_22may.pt")

def get_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    iou = interArea / float(boxAArea + boxBArea - interArea)
    return iou


def inside_iou(det_box, parent_box, return_iou=False):
    xi1 = max(det_box[0], parent_box[0])
    yi1 = max(det_box[1], parent_box[1])
    xi2 = min(det_box[2], parent_box[2])
    yi2 = min(det_box[3], parent_box[3])
    inter_area = max(xi2 - xi1, 0) * max(yi2 - yi1, 0)

    det_area = (det_box[2] - det_box[0]) * (det_box[3] - det_box[1])
    iou = inter_area / float(det_area) if det_area != 0 else 0
    if return_iou:
        return iou
    else:
        return iou > 0.5

def get_iou_polygon(polygonA, polygonB):
    # This function would calculate the IoU between two polygons.
    polyA = Polygon([(polygonA[i], polygonA[i + 1]) for i in range(0, len(polygonA), 2)])
    polyB = Polygon([(polygonB[i], polygonB[i + 1]) for i in range(0, len(polygonB), 2)])
    iou = polyA.intersection(polyB).area / polyA.union(polyB).area
    return iou

def nms_classwise(df):
    def nms_single_class(df_class):
        elements = df_class.to_dict('records')
        elements.sort(key=lambda x: x['confidence'], reverse=True)
        selected_elements = []

        for current in elements:
            # Correctly using list comprehension for creating a list of vertices for the current polygon
            current_polygon = [val for i in range(4) for val in (current[f'x{i+1}'], current[f'y{i+1}'])]

            overlap = False
            for selected in selected_elements:
                # Similarly constructing the polygon vertices list for the selected element
                selected_polygon = [val for i in range(4) for val in (selected[f'x{i+1}'], selected[f'y{i+1}'])]

                if get_iou_polygon(current_polygon, selected_polygon) > 0.3:  # Using the corrected polygon IoU function
                    overlap = True
                    break

            if not overlap:
                selected_elements.append(current)

        return pd.DataFrame(selected_elements)

    grouped = df.groupby('label')
    nms_dfs = [nms_single_class(group) for _, group in grouped]
    return pd.concat(nms_dfs, ignore_index=True)
def get_yolo_boxes(image_path, cropped_img = None, conf_thresh = .25, estimate_rotation = False, dump_images = False):
    try:
        im = np.array(cropped_img)
        im = im[:, :, ::-1].copy()
    except:
        im = cv2.imread(image_path)
    pageheight, pagewidth= im.shape[0], im.shape[1]
    predictions = det_model(im, imgsz=800, conf=0.05, max_det=10000, verbose=False)
    xywhr =  predictions[0].obb.xywhr.tolist()
    xyxyxyxy = predictions[0].obb.xyxyxyxy.tolist()
    xyxy = predictions[0].obb.xyxy.tolist()
    classes = predictions[0].obb.cls.tolist()
    confidence = predictions[0].obb.conf.tolist()
    names = [
        "box", "table", "column", "header",
        "signature", "figure", "paragraph", "logo", "kv", "stamp"
    ] # Get class names
    name_to_id = {name: i for i, name in enumerate(names)}

    # Color codes for each label
    color_codes = {
        "box": (255, 0, 0),  # Blue
        "header": (255, 0, 0),  # Blue
        "table": (0, 255, 0),  # Green
        "column": (0, 255, 0),  # Green
        "logo": (0, 0, 255),  # Red
        "signature": (0, 255, 255),  # Yellow
        "stamp": (0, 0, 0),  # Black
        # Default color for other labels (optional)
        "default": (255, 255, 255)  # White
    }
    if dump_images:
        # Check if results directory exists, if not, create it
        if not os.path.exists('results'):
            os.makedirs('results')
        # Make a copy of the image to draw on
        im_copy = im.copy()
    df_data = []
    for i, a in enumerate(xywhr):
        centroid_x, centroid_y, width, height, radians = a
        x1, y1 = int(xyxyxyxy[i][0][0]), int(xyxyxyxy[i][0][1])
        x2, y2 = int(xyxyxyxy[i][1][0]), int(xyxyxyxy[i][1][1])
        x3, y3 = int(xyxyxyxy[i][2][0]), int(xyxyxyxy[i][2][1])
        x4, y4 = int(xyxyxyxy[i][3][0]), int(xyxyxyxy[i][3][1])

        label_id = int(classes[i])
        label_name = names[label_id]
        conf = confidence[i]

        if conf > conf_thresh:
            df_data.append({
                "x1": int(x1), "y1": int(y1),
                "x2": int(x2), "y2": int(y2),
                "x3": int(x3), "y3": int(y3),
                "x4": int(x4), "y4": int(y4),
                "centroid_x": centroid_x, "centroid_y": centroid_y,
                "width": width, "height": height, "angle": radians * 180 / 3.14159,
                "label": label_name, "confidence": conf
            })

    df = pd.DataFrame(df_data, columns=['x1', 'y1', 'x2', 'y2', 'x3', 'y3', 'x4', 'y4', 'centroid_x', 'centroid_y', 'width', 'height', 'angle', 'label', 'confidence'])

    if len(df) > 0:
        filtered_df = nms_classwise(df)
        if dump_images and len(df) > 0:
        # Iterate over each row in the filtered DataFrame to draw the polygons
            for index, row in filtered_df.iterrows():
            # Extract points for the current polygon
                pts = np.array([(row['x1'], row['y1']), (row['x2'], row['y2']),
                            (row['x3'], row['y3']), (row['x4'], row['y4'])], np.int32)
            # Reshape points in the form required by polylines
                pts = pts.reshape((-1, 1, 2))

            # Use specified color or default if not specified
                color = color_codes.get(row['label'], color_codes["default"])

            # Draw the polygon on the image
                cv2.polylines(im_copy, [pts], isClosed=True, color=color, thickness=2)

        # Save the modified image
            output_image_path = os.path.join('results', os.path.basename(image_path))
            cv2.imwrite(output_image_path, im_copy)
    else:
        filtered_df = df
    final_df = pd.DataFrame(columns=['x1', 'y1', 'x2', 'y2', 'label', 'confidence'])
    rows = []
    if not filtered_df.empty:
        for index, row in filtered_df.iterrows():
            # Calculate min and max values of x and y
            x_min = min(row['x1'], row['x2'], row['x3'], row['x4'])
            x_max = max(row['x1'], row['x2'], row['x3'], row['x4'])
            y_min = min(row['y1'], row['y2'], row['y3'], row['y4'])
            y_max = max(row['y1'], row['y2'], row['y3'], row['y4'])

            # Append transformed row to the list
            rows.append({'x1': x_min, 'y1': y_min, 'x2': x_max, 'y2': y_max,
                        'label': row['label'], 'confidence': row['confidence']})

    # Create DataFrame from the collected rows
    final_df = pd.DataFrame(rows)

    return final_df


def extract_textract(image_path, mode = ['LINE'], access_key_id = access_key, secret_access_key = secret):
    try:
        textract = boto3.client('textract',
                                aws_access_key_id=access_key_id,
                                aws_secret_access_key=secret_access_key,
                                region_name='us-east-1')  # Make sure to replace 'us-east-1' with your actual AWS region
        with open(image_path, 'rb') as document:
            img_bytes = document.read()
        img = Image.open(image_path)
        width, height = img.size
        response = textract.detect_document_text(Document={'Bytes': img_bytes})
        data = []
        for block in response['Blocks']:
            if block['BlockType'] in mode:
                # Extract the bounding box
                bb = block['Geometry']['BoundingBox']
                x1 = int(bb['Left'] * width)
                y1 = int(bb['Top'] * height)
                x2 = int(x1 + bb['Width'] * width)
                y2 = int(y1 + bb['Height'] * height)
                text = block.get('Text', '')
                confidence = block.get('Confidence', 0)
                data.append({'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2, 'text': text, 'confidence': confidence})
        df = pd.DataFrame(data, columns=['x1', 'y1', 'x2', 'y2', 'text'])
        return df
    except Exception as e:
        return f"An error occurred: {str(e)}"

def filter_boundingbox_output(df_textract):
    # Convert DataFrame rows to a list of dictionaries for easier manipulation
    textract_data = df_textract.to_dict('records')
    filtered_boxes = []
    skip_indices = set()
    for i, box_a in enumerate(textract_data):
        if i in skip_indices:  # Skip this box if it's already marked as contained within another
            continue
        for j, box_b in enumerate(textract_data):
            if i != j and j not in skip_indices:
                # Check if box_b is completely inside box_a
                if inside_iou([box_b['x1'], box_b['y1'], box_b['x2'], box_b['y2']],
                              [box_a['x1'], box_a['y1'], box_a['x2'], box_a['y2']], return_iou=True) == 1:
                    # Mark box_b to be skipped in future iterations
                    skip_indices.add(j)
        # After checking all boxes for containment, add the current box to the filtered list
        filtered_boxes.append(box_a)
    # Convert the filtered list of boxes back to a DataFrame
    df_textract_filtered = pd.DataFrame(filtered_boxes)
    return df_textract_filtered

def calculate_composite_dimensions(boxes, horizontal_padding, vertical_buffer, font_size, text_width=100, gap = 5):
    """
    Calculate the dimensions for the composite image, including additional space for text.
    """
    # Load the font and calculate text size for a representative string
    font = ImageFont.truetype("arial.ttf", font_size)
    text_height = max(textsize(f"{box[0]}, {box[1]}, {box[2]}, {box[3]}", font=font)[1] for box in boxes)

    max_padded_width = max(box[2] - box[0] for box in boxes) + 2 * horizontal_padding + text_width
    total_height = sum(box[3] - box[1] + text_height + gap for box in boxes)

    return max_padded_width, total_height

def textsize(text, font):
    im = Image.new(mode="P", size=(0, 0))
    draw = ImageDraw.Draw(im)
    _, _, width, height = draw.textbbox((0, 0), text=text, font=font)
    return width, height
def get_median_crop_color(crops):
    # Convert crops to grayscale and flatten the list of pixel values
    pixels = np.concatenate([np.array(crop.convert('L')).flatten() for crop in crops])
    # Calculate the median pixel value
    median_pixel = int(np.median(pixels))
    return (median_pixel, median_pixel, median_pixel)  # Return as RGB for compatibility

def parse_extracted_text(extracted_text):
    rows = []
    for line in extracted_text:
        # Remove any variations of colons from the line
        line = re.sub(r'\:+', ':', line)

        # Extract numbers and assume the first four are coordinates
        numbers = re.findall(r'\d+', line)
        text_part = ""

        if len(numbers) >= 4:
            # Assume the first four numbers are coordinates
            coordinates = numbers[:4]
            # Find the start index of the text, which is after the last digit of the fourth coordinate
            pattern = re.compile(r'(\d+)')
            matches = list(pattern.finditer(line))
            if len(matches) >= 4:
                # The text starts right after the fourth number
                text_start_index = matches[3].end()
                text_part = line[text_start_index:].strip()
                # Remove any leading characters that are not alphabetic or numeric
                text_part = re.sub(r'^[^a-zA-Z0-9]*', '', text_part)
        else:
            # Default to zeros if there aren't enough numbers
            coordinates = ['0', '0', '0', '0']
            text_part = line

        # Parse coordinates as integers
        coords = list(map(int, coordinates))

        # Create a dictionary for the row
        row = {'x1': coords[0], 'y1': coords[1], 'x2': coords[2], 'y2': coords[3], 'text': text_part}
        rows.append(row)

    return pd.DataFrame(rows)

def crop_and_extract_text(image_path, boxes, gap=0, horizontal_padding=10, vertical_buffer=10, minimal_gap=5, target_text_height=30):
    def consolidate_rows(df):
        used_boxes = set()
        box_to_row_data = {}
        anchor_boxes = df[df['x2'] - df['x1'] > 200].copy()  # Filter anchor boxes based on width
        for index, anchor in anchor_boxes.iterrows():
            y1, y2 = anchor['y1'], anchor['y2']
            for idx, box in df.iterrows():
                if idx in used_boxes:
                    continue
                iou = calculate_iou((0, y1, 1, y2), (0, box['y1'], 1, box['y2']))
                if iou > box_to_row_data.get(idx, (0, None))[0]:  # Update only if this iou is greater
                    box_to_row_data[idx] = (iou, index)  # Store iou and row index of the anchor
                    used_boxes.add(idx)

        # Combine text components based on identified overlaps with anchor boxes
        results = []
        for idx, (_, anchor_idx) in box_to_row_data.items():
            text = df.loc[idx, 'text']
            x1, y1, x2, y2 = df.loc[anchor_idx, ['x1', 'y1', 'x2', 'y2']]
            results.append({'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2, 'text': text})

        final_df = pd.DataFrame(results)
        final_df = final_df.groupby(['x1', 'y1', 'x2', 'y2']).agg({'text': ' '.join}).reset_index()
        return final_df

    original_img = Image.open(image_path)
    crops = [original_img.crop((box[0], box[1], box[2], box[3])) for box in boxes]
    median_color = get_median_crop_color(crops)
    font = ImageFont.truetype("arial.ttf", target_text_height)  # Use target_text_height as an initial guess
    fontwidth, fontheight = textsize('100, 100, 100, 100', font=font)
    # Prepare the composite image
    max_width, total_height = calculate_composite_dimensions(boxes, horizontal_padding, vertical_buffer, target_text_height, fontwidth)
    max_width = int(max_width * 1.2)
    composite_img = Image.new('RGB', (max_width, total_height), median_color)
    draw = ImageDraw.Draw(composite_img)
    current_y = vertical_buffer
    for i, (box, crop) in enumerate(zip(boxes, crops)):
        enhancer = ImageEnhance.Contrast(crop)
        enhanced_img = enhancer.enhance(2)
        padded_img = ImageOps.expand(enhanced_img, border=(horizontal_padding, vertical_buffer), fill=median_color)
        crop_width, crop_height = padded_img.size
        # Text preparation
        text = f"{box[0]}, {box[1]}, {box[2]}, {box[3]} :"
        text_size = textsize(text, font=font)
        text_x = minimal_gap
        crop_y_centroid = current_y + crop_height // 2
        text_y = crop_y_centroid - text_size[1] // 2
        draw.text((text_x, text_y), text, fill=(0, 0, 0), font=font)
        crop_x = text_size[0] + minimal_gap
        composite_img.paste(padded_img.convert('L'), (crop_x, current_y))
        current_y += crop_height + gap
    temp_path = 'temp_composite.jpg'
    composite_img.save(temp_path)
    # OCR to extract text including coordinates
    # df_text = extract_textract(temp_path, mode=['LINE'])
    df_text = extract_textract(temp_path)
    df_text = consolidate_rows(df_text)
    extracted_text = df_text['text'].tolist()
    parsed_text = parse_extracted_text(extracted_text)
    return pd.DataFrame(parsed_text)


def match_and_draw(image_path, df_textract, df_yolo_orig,  output_folder,mode='draw', save_csv = True):

    df_textract = filter_boundingbox_output(df_textract)
    relevant_classes = ['box', 'signature', 'header']
    df_yolo = filter_boundingbox_output(df_yolo_orig[df_yolo_orig['label'].isin(relevant_classes)])
    df_yolo_kv = filter_boundingbox_output(df_yolo_orig[df_yolo_orig['label'] == 'kv'])
    df_yolo_table = filter_boundingbox_output(df_yolo_orig[df_yolo_orig['label'].isin(['table', 'column'])])
    combined_data = []
    matched_yolo_indices = set()  # Use a set for efficient look-up
    unmatched_textract_indices = set(df_textract.index)  # Initialize with all textract indices
    unmatched_yolo_boxes = []
    raw_image = cv2.imread(image_path)
    pageheight, pagewidth = raw_image.shape[0], raw_image.shape[1]
    if mode == 'draw':
        original_image = raw_image.copy()
        textract_image = raw_image.copy()
        yolo_image = raw_image.copy()

        for _, row in df_textract.iterrows():
            cv2.rectangle(textract_image, (row['x1'], row['y1']), (row['x2'], row['y2']), (0, 0, 255), 2)

        for _, row in df_yolo.iterrows():
            if row['label'] in relevant_classes:
                cv2.rectangle(yolo_image, (row['x1'], row['y1']), (row['x2'], row['y2']), (255, 0, 0), 2)

        for _, row in df_yolo_table.iterrows():
            cv2.rectangle(original_image, (row['x1'], row['y1']), (row['x2'], row['y2']), (125, 0, 125), 3)

    for index_y, row_y in df_yolo.iterrows():
        max_iou = 0
        best_match_index_t = None
        for index_t, row_t in df_textract.iterrows():
            iou = get_iou([row_t['x1'], row_t['y1'], row_t['x2'], row_t['y2']],
                          [row_y['x1'], row_y['y1'], row_y['x2'], row_y['y2']])
            iou_height = get_iou([0, row_t['y1'], 1, row_t['y2']],
                          [0, row_y['y1'], 1, row_y['y2']])
            if iou > max_iou:
                max_iou = iou
                best_match_index_t = index_t
        if max_iou > 0.25:
        # if max_iou > 0:
            matched_yolo_indices.add(index_y)
            if best_match_index_t is not None:
                unmatched_textract_indices.discard(best_match_index_t)
                combined_data.append({
                    "yolo_coords": [row_y['x1'], row_y['y1'], row_y['x2'], row_y['y2']],
                    "textract_coords": [[df_textract.loc[best_match_index_t][col] for col in ['x1', 'y1', 'x2', 'y2']]],
                    "textract_texts": [df_textract.loc[best_match_index_t]['text']]
                })
            # Draw green boxes for matched YOLO boxes
            if mode == 'draw':
                cv2.rectangle(original_image, (row_y['x1'], row_y['y1']), (row_y['x2'], row_y['y2']), (0, 255, 0), 2)
    # Draw Blue boxes for unmatched YOLO boxes of relevant classes
    for index_y, row_y in df_yolo.iterrows():
        if (index_y not in matched_yolo_indices and row_y['label'] in relevant_classes):
            unmatched_yolo_boxes.append([row_y['x1'], row_y['y1'], row_y['x2'], row_y['y2']])
            if mode == 'draw':
                cv2.rectangle(original_image, (row_y['x1'], row_y['y1']), (row_y['x2'], row_y['y2']), (255, 0, 0), 2)

    for index_y, row_y in df_yolo_kv.iterrows():
        x1, y1, x2, y2 = row_y['x1'], row_y['y1'], row_y['x2'], row_y['y2']
        if y1 < pageheight / 3 and (x1+x2)/2 > 2*pagewidth/3:
            x2 = max(x2,pagewidth-10)
            unmatched_yolo_boxes.append([x1, y1, x2, y2])

        if mode == 'draw':
            cv2.rectangle(original_image, (row_y['x1'], row_y['y1']), (row_y['x2'], row_y['y2']), (255, 0, 0), 2)

    missed_boxes = []
    for box in unmatched_yolo_boxes:
        # Modify extract_text_from_image to suit this specific use case
        missed_boxes.append(box)
    if missed_boxes:
        # Call the function to extract text for the missed boxes
        df_missed_text = crop_and_extract_text(image_path, missed_boxes)

        # Re-match the newly extracted texts with the YOLO boxes based on IOU
        for index_y, row_y in df_yolo.iterrows():
            if row_y['label'] in ['box', 'signature', 'header'] and index_y not in matched_yolo_indices:
                max_iou = 0
                best_match_index_t = None
                for index_t, row_t in df_missed_text.iterrows():
                    iou = get_iou([row_t['x1'], row_t['y1'], row_t['x2'], row_t['y2']],
                                [row_y['x1'], row_y['y1'], row_y['x2'], row_y['y2']])
                    if iou > max_iou:
                        max_iou = iou
                        best_match_index_t = index_t

                if max_iou > 0:
                    matched_yolo_indices.add(index_y)
                    combined_data.append({
                        "yolo_coords": [row_y['x1'], row_y['y1'], row_y['x2'], row_y['y2']],
                        "textract_coords": [[df_missed_text.loc[best_match_index_t][col] for col in ['x1', 'y1', 'x2', 'y2']]],
                        "textract_texts": [df_missed_text.loc[best_match_index_t]['text']]
                    })
        for index_y, row_y in df_yolo_kv.iterrows():
            if row_y['label'] == 'kv':
                max_iou = 0
                best_match_index_t = None
                for index_t, row_t in df_missed_text.iterrows():
                    iou = get_iou([row_t['x1'], row_t['y1'], row_t['x2'], row_t['y2']],
                                [row_y['x1'], row_y['y1'], row_y['x2'], row_y['y2']])
                    if iou > max_iou:
                        max_iou = iou
                        best_match_index_t = index_t
                if max_iou > 0:
                    matched_yolo_indices.add(index_y)
                    if 'Code' in df_missed_text.loc[best_match_index_t]['text']:
                        ocr_text = df_missed_text.loc[best_match_index_t]['text']
                        combined_data.append({
                            "yolo_coords": [row_y['x1'], row_y['y1'], row_y['x2'], row_y['y2']],
                            "textract_coords": [[df_missed_text.loc[best_match_index_t][col] for col in ['x1', 'y1', 'x2', 'y2']]],
                            "textract_texts": [ocr_text]
                        })
    if mode == 'draw':
        for data in combined_data:
            # Each entry in combined_data should have 'textract_coords' and 'textract_texts'
            for coords, text in zip(data['textract_coords'], data['textract_texts']):
                # Ensuring coords is a list with 4 elements [x1, y1, x2, y2]
                if isinstance(coords, list) and len(coords) == 4:
                    x1, y1, x2, y2 = coords
                    position = (max(x1 - 10, 0), max((y1+y2)//2, 0))
                    cv2.putText(original_image, text, position, cv2.FONT_HERSHEY_SIMPLEX, .7, (0, 125, 125), 1)
        h_min = min(textract_image.shape[0], yolo_image.shape[0], original_image.shape[0])
        textract_image = cv2.resize(textract_image, (int(textract_image.shape[1] * h_min / textract_image.shape[0]), h_min))
        yolo_image = cv2.resize(yolo_image, (int(yolo_image.shape[1] * h_min / yolo_image.shape[0]), h_min))
        original_image = cv2.resize(original_image, (int(original_image.shape[1] * h_min / original_image.shape[0]), h_min))
        final_image = cv2.hconcat([textract_image, yolo_image, original_image])

        # Save the combined image
        output_image_path = 'results/' + os.path.basename(image_path).replace('.jpg', 'combined.jpg')
        cv2.imwrite(output_image_path, final_image)

    combined_data = pd.DataFrame(combined_data)  # Convert combined_data to DataFrame for easier manipulation
    df_yolo_orig['text'] = None  # Initialize 'text' column with None
    for index, row in combined_data.iterrows():
        mask = (df_yolo_orig['x1'] == row['yolo_coords'][0]) & \
               (df_yolo_orig['y1'] == row['yolo_coords'][1]) & \
               (df_yolo_orig['x2'] == row['yolo_coords'][2]) & \
               (df_yolo_orig['y2'] == row['yolo_coords'][3])

        df_yolo_orig.loc[mask, 'text'] = row['textract_texts'][0]

    # output_csv_path = 'results/' + os.path.basename(image_path).replace('.jpg', '.csv')
    output_csv_path = output_folder + os.path.basename(image_path).replace('.jpg', '.csv')

    if save_csv:
        df_yolo_orig.to_csv(output_csv_path)
    return df_yolo_orig



/opt/conda/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/lib/python3.8/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [3]:

def create_bottom_table(df_ocr):
    def filter_boxes(column, all_boxes, bottom_table, column_number):
        boxes = all_boxes[((all_boxes['x1'] + all_boxes['x1'])/2 >= column['x1']) & ((all_boxes['x1'] + all_boxes['x1'])/2 <= column['x2'])].copy()
        col_boxes = [column['x1'], min(bottom_table['y1'], column['y1']), column['x2'], max(bottom_table['y2'], column['y2'])]
        boxes.loc[:, 'iou'] = boxes.apply(lambda row: calculate_iou_inside((col_boxes[0], col_boxes[1], col_boxes[2], col_boxes[3]), (row['x1'], row['y1'], row['x2'], row['y2'])), axis=1)
        boxes = boxes[boxes['iou'] > .5]
        return boxes.sort_values(by='y1')
    data = df_ocr
    tables = data[data['label'] == 'table'].sort_values(by='y1')
    bottom_table = tables.iloc[-1] if not tables.empty else None
    box_data = data[(data['label'] == 'box') | (data['label'] == 'signature')].copy()
    if bottom_table is not None:
        bottom_table_box = (bottom_table['x1'], bottom_table['y1'], bottom_table['x2'], bottom_table['y2'])
        columns = data[data['label'] == 'column'].copy()
        columns.loc[:, 'iou'] = columns.apply(lambda row: calculate_iou(bottom_table_box, (row['x1'], row['y1'], row['x2'], row['y2'])), axis=1)
        columns.loc[:, 'y_centroid'] = columns.apply(lambda row: (row['y1'] + row['y2']) /2, axis=1)
        columns = columns[(columns['iou'] > 0.1) & (columns['y_centroid'] > bottom_table_box[1])].sort_values(by='x1')
        if len(columns) < 4:
            return None
        column_boxes = [filter_boxes(columns.iloc[i], box_data, bottom_table, i) for i in range(len(columns))]
        rows = []
        used_boxes = set()
        box_to_row_data = {}
        row_confidences = []
        # Initialize each row with placeholders for all columns
        column_count = len(columns)
        max_row_len = max(len(column_boxes[0]), len(column_boxes[1]))
        if len(column_boxes[0]) == max_row_len:
            ref_col = 0
        else:
            ref_col = 1
        for i in range(max_row_len):  # Evaluate each reference row from the first column
            y1 = column_boxes[ref_col].iloc[i]['y1']
            y2 = column_boxes[ref_col].iloc[i]['y2']
            for j in range(column_count):  # Iterate over each column
                for index, box in column_boxes[j].iterrows():
                    if index in used_boxes:
                        continue
                    iou = calculate_iou((0, y1, 1, y2), (0, box['y1'], 1, box['y2']))
                    if iou > box_to_row_data.get((index, j), (0, None))[0]:  # Update only if this iou is greater
                        box_to_row_data[(index, j)] = (iou, i)  # Store iou and row index
        for (index, col_index), (iou, row_index) in box_to_row_data.items():
            if col_index == 4:
                iou_thresh = .01
            elif col_index == ref_col:
                iou_thresh = .1
            else:
                iou_thresh = .2
            if iou > iou_thresh:
                selected_text = column_boxes[col_index].loc[index, 'text']
                confidence = column_boxes[col_index].loc[index, 'confidence']
                if row_index >= len(rows):
                    while row_index >= len(rows):
                        rows.append({f'Column {k + 1}': '' for k in range(column_count)})  # Initialize all columns
                        row_confidences.append([])
            try:
                max_iou = rows[row_index].get(f'MaxIoU{col_index + 1}', 0)
            except:
                max_iou = 0
            if iou > iou_thresh and iou > max_iou:  # Only consider this box if its IoU is greater than the current max IoU
                selected_text = column_boxes[col_index].loc[index, 'text']
                confidence = column_boxes[col_index].loc[index, 'confidence']
                # Update the row data and max IoU for this cell
                rows[row_index][f'Column {col_index + 1}'] = selected_text
                rows[row_index][f'MaxIoU{col_index + 1}'] = iou  # Update the maximum IoU for this cell
                row_confidences[row_index].append(confidence)
                used_boxes.add(index)
        for row in rows:
            for col_index in range(1, column_count + 1):
                row.pop(f'MaxIoU{col_index}', None)
        bottom_table_df = pd.DataFrame(rows)
        bottom_table_df['confidence'] = row_confidences
        return bottom_table_df

    return None
def postprocess_bottom_table(df):
    def match_template(df):
        # List of valid political party entries transformed to lowercase for flexible matching
        valid_parties = [
            "A", "AA", "AAC", "ADC", "ADP", "APC", "APGA", "APM", "APP",
            "BP", "LP", "NNPP", "NRM", "PDP", "SDP", "YPP", "ZLP"
        ]

        # Prepare 'Political Party' entries by stripping spaces and converting to uppercase
        df['Political Party'] = df['Political Party'].apply(lambda x: x.strip().upper() if isinstance(x, str) else x)

        # Handle partial matching for "PDP"
        for i in range(1, len(df) - 1):
            current_party = df.at[i, 'Political Party']
            if isinstance(current_party, str) and current_party.startswith("P") and current_party.endswith("P"):
                prev_party = df.at[i - 1, 'Political Party'] if i > 0 else ''
                next_party = df.at[i + 1, 'Political Party'] if i < len(df) - 1 else ''
                if prev_party == "NRM" and next_party == "PRP":
                    df.at[i, 'Political Party'] = "PDP"

        # Filter the DataFrame for rows where 'Political Party' is in the list of valid parties
        filtered_df = df[df['Political Party'].isin(valid_parties)]

        return filtered_df

    # Define the search areas and target keywords for each column rename action
    search_areas = {
        'Political Party': {
            'keywords': ['political', 'party'],
            'columns': [0, 1],
            'new_name': 'Political Party'
        },
        'Vote Score (Figure)': {
            'keywords': ['figure'],
            'columns': [1, 2, 3],
            'new_name': 'Vote Score (Figure)'
        },
        'Vote Score (Word)': {
            'keywords': ['word'],
            'columns': [1, 2, 3],
            'new_name': 'Vote Score (Word)'
        },
        'Signature': {
            'keywords': ['polling', 'signature'],
            'columns': [3],
            'new_name': 'Signature'
        }
    }
    # Iterate through defined search areas and apply substring checking
    for name, details in search_areas.items():
        for col_index in details['columns']:
            if col_index < len(df.columns):
                column = df.columns[col_index]
                for i in range(min(5, len(df))):
                    cell_value = str(df.loc[i, column]).lower()
                    if any(keyword in cell_value for keyword in details['keywords']):
                        df.rename(columns={column: details['new_name']}, inplace=True)
                        break

    # Convert the 'Signature' column to True if any string present has alphabets
    if 'Signature' in df.columns:
        df['Signature'] = df['Signature'].apply(lambda x: True if any(c.isalpha() for c in str(x)) else None)

    return match_template(df)


In [4]:


def calculate_iou_inside(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return interArea / min(boxAArea, boxBArea)


def calculate_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return interArea / float(boxAArea + boxBArea - interArea)

def create_top_table(df_ocr, mode='missing_top'):
    def filter_boxes(column, all_boxes, bottom_table):
        boxes = all_boxes
        col_boxes = [column['x1'], min(bottom_table['y1'], column['y1']), column['x2'], max(bottom_table['y2'], column['y2'])]
        boxes.loc[:, 'iou'] = boxes.apply(lambda row: calculate_iou_inside((col_boxes[0], col_boxes[1], col_boxes[2], col_boxes[3]), (row['x1'], row['y1'], row['x2'], row['y2'])), axis=1)
        boxes = boxes[boxes['iou'] > .25]
        return boxes.sort_values(by='y1')

    data = df_ocr
    tables = data[data['label'] == 'table'].sort_values(by='y1')
    bottom_table = tables.iloc[0] if not tables.empty else None
    box_data = data[(data['label'] == 'box')].copy()
    if bottom_table is not None:
        bottom_table_box = (bottom_table['x1'], bottom_table['y1'], bottom_table['x2'], bottom_table['y2'])
        columns = data[data['label'] == 'column'].copy()
        columns.loc[:, 'iou'] = columns.apply(lambda row: calculate_iou(bottom_table_box, (row['x1'], row['y1'], row['x2'], row['y2'])), axis=1)
        columns.loc[:, 'y_centroid'] = columns.apply(lambda row: (row['y1'] + row['y2']) / 2, axis=1)
        columns = columns[(columns['iou'] > 0.1) & (columns['y_centroid'] > bottom_table_box[1])].sort_values(by='x1')
        column_boxes = [filter_boxes(columns.iloc[i], box_data, bottom_table) for i in range(len(columns))]
        rows = []
        used_boxes = set()
        box_to_row_data = {}
        entity_max_iou = {}
        row_confidences = []
        # Collect all IoU values for each entity and store them in a dictionary
        for i in range(len(column_boxes[0])):  # Evaluate each reference row from the first column
            y1 = column_boxes[0].iloc[i]['y1']
            y2 = column_boxes[0].iloc[i]['y2']
            for j in range(len(column_boxes)):  # Iterate over each column
                for index, box in column_boxes[j].iterrows():
                    if index in used_boxes:
                        continue
                    iou = calculate_iou((0, y1, 1, y2), (0, box['y1'], 1, box['y2']))
                    if iou > 0.1:
                        if index not in box_to_row_data:
                            box_to_row_data[index] = []
                        box_to_row_data[index].append((iou, j, i))  # Store iou, col_index, row_index

        # Sort assignments by IoU in descending order
        sorted_assignments = sorted([(iou, index, col_index, row_index) for index in box_to_row_data for (iou, col_index, row_index) in box_to_row_data[index]], key=lambda x: -x[0])

        # Initialize each row with placeholders for all columns
        column_count = len(columns)
        for i in range(len(column_boxes[0])):  # Evaluate each reference row from the first column
            if len(rows) <= i:
                rows.append({f'Column {k + 1}': '' for k in range(column_count)})  # Initialize all columns for the row
                row_confidences.append([])

        # Assign each entity to the row with the highest IoU, ensuring no reassignment
        assigned_entities = set()
        for iou, index, col_index, row_index in sorted_assignments:
            if index not in assigned_entities:
                selected_text = column_boxes[col_index].loc[index, 'text']
                confidence = column_boxes[col_index].loc[index, 'confidence']
                if not rows[row_index][f'Column {col_index + 1}']:  # Only assign if the slot is empty
                    rows[row_index][f'Column {col_index + 1}'] = selected_text
                    row_confidences[row_index].append(confidence)
                    assigned_entities.add(index)

        # Mode 'missing_top': Ensure the topmost entry in Column 2 is present, if not, insert it at the top and shift others down
        if mode == 'missing_top':
            topmost_entry = None
            topmost_index = -1
            for index, box in column_boxes[1].iterrows():
                if topmost_entry is None or box['y1'] < topmost_entry['y1']:
                    topmost_entry = box
                    topmost_index = index

            if topmost_entry is not None and topmost_index not in assigned_entities:
                topmost_text = topmost_entry['text']
                topmost_confidence = topmost_entry['confidence']
                rows.insert(0, {f'Column {k + 1}': '' for k in range(column_count)})  # Insert a new row at the top
                row_confidences.insert(0, [])
                rows[0][f'Column 2'] = topmost_text  # Assign topmost text to the new top row
                row_confidences[0].append(topmost_confidence)
                assigned_entities.add(topmost_index)

        bottom_table_df = pd.DataFrame(rows)
        bottom_table_df['confidence'] = row_confidences
        return bottom_table_df

    return None


def align_rows_old(df, template):
    # Map each template to the first occurrence index in the DataFrame
    template_indices = {t: df[df['Matched Template'] == t].index.min() for t in template if (df['Matched Template'] == t).any()}

    # Find the first template index and check for misplaced 'Votes' above it
    if template_indices and template[0] in template_indices:
        first_template_index = template_indices[template[0]]
        misplaced_votes_indices = df.iloc[:first_template_index]['Votes'][
            df.iloc[:first_template_index]['Votes'].apply(
                lambda x: isinstance(x, str) and len(x.split()) > 0 and x.split()[0].isdigit() and int(x.split()[0]) >= 20
            )
        ]
        # If there are misplaced votes, initiate the push down operation
        if not misplaced_votes_indices.empty:
            misplaced_vote = misplaced_votes_indices.iloc[0]  # Take the first misplaced vote
            misplaced_confidence = df.at[misplaced_votes_indices.index[0], 'confidence']
            # Start pushing down from the first template index
            for i in range(first_template_index, len(df)):
                if i in template_indices.values():  # Only push down within template matched indices
                    if pd.isna(df.at[i, 'Votes']) or df.at[i, 'Votes'].strip() == '':  # Stop if next is blank
                        df.at[i, 'Votes'] = misplaced_vote
                        df.at[i, 'confidence'] = misplaced_confidence
                        break  # Halt pushing down when encountering the first blank after a template match
                    current_vote = df.at[i, 'Votes']
                    current_confidence = df.at[i, 'confidence']
                    df.at[i, 'Votes'] = misplaced_vote
                    df.at[i, 'confidence'] = misplaced_confidence
                    misplaced_vote = current_vote
                    misplaced_confidence = current_confidence
    return df
def align_rows(df, template):
    # Map each template to the first occurrence index in the DataFrame
    template_indices = {t: df[df['Matched Template'] == t].index.min() for t in template if (df['Matched Template'] == t).any()}

    # Find the first template index and check for misplaced 'Votes' above it
    if template_indices and template[0] in template_indices:
        first_template_index = template_indices[template[0]]

        # Detect potential duplicates above the first template index
        for i in range(first_template_index - 1):
            current_vote = df.at[i, 'Votes']
            next_vote = df.at[i + 1, 'Votes']
            if isinstance(current_vote, str) and isinstance(next_vote, str):
                current_split = current_vote.split()
                next_split = next_vote.split()
                if current_split and next_split:
                    current_digits = ''.join(filter(str.isdigit, current_split[0])) if current_split[0].isdigit() else ''
                    next_digits = ''.join(filter(str.isdigit, next_split[0])) if next_split[0].isdigit() else ''
                    if (current_vote == next_vote) or (current_digits == next_digits and current_digits != ''):
                        # Mark the next row's Votes as a potential blank if a duplicate is found
                        df.at[i + 1, 'Votes'] = ''
                        df.at[i + 1, 'confidence'] = []
                        break  # Only remove one, the topmost potential blank

        misplaced_votes_indices = df.iloc[:first_template_index]['Votes'][
            df.iloc[:first_template_index]['Votes'].apply(
                lambda x: isinstance(x, str) and len(x.split()) > 0 and x.split()[0].isdigit() and int(x.split()[0]) >= 20
            )
        ]

        # If there are misplaced votes, initiate the push down operation
        if not misplaced_votes_indices.empty:
            misplaced_vote = misplaced_votes_indices.iloc[0]  # Take the first misplaced vote
            misplaced_confidence = df.at[misplaced_votes_indices.index[0], 'confidence']

            # Start pushing down from the first template index
            for i in range(first_template_index, len(df)):
                if i in template_indices.values():  # Only push down within template matched indices
                    if pd.isna(df.at[i, 'Votes']) or df.at[i, 'Votes'].strip() == '':  # Stop if next is blank
                        df.at[i, 'Votes'] = misplaced_vote
                        df.at[i, 'confidence'] = misplaced_confidence
                        break  # Halt pushing down when encountering the first blank after a template match
                    current_vote = df.at[i, 'Votes']
                    current_confidence = df.at[i, 'confidence']
                    df.at[i, 'Votes'] = misplaced_vote
                    df.at[i, 'confidence'] = misplaced_confidence
                    misplaced_vote = current_vote
                    misplaced_confidence = current_confidence

    return df
def match_top_table(df):
    template = [
        "1. Number of Voters on the Register",
        "2. Number of Accredited Voters",
        "3. Number of Ballot Papers Issued to the Polling Units",
        "4. Number of Unused Ballot Papers",
        "5. Number of Spoiled Ballot papers",
        "6. Number of Rejected Ballots",
        "7. Number of Total Valid Votes (Total valid Votes cast for all parties)",
        "8. Total Number of Used Ballot Papers (Total of #5 + #6 + #7 above)"
    ]

    if df is not None and not df.empty and 'Column 1' in df.columns:
        # Extract the closest match from the template along with the score
        df['Matched Template and Score'] = df['Column 1'].apply(
            lambda x: process.extractOne(x, template, scorer=fuzz.token_sort_ratio, score_cutoff=70)
        )
        df['Matched Template'] = df['Matched Template and Score'].apply(lambda x: x[0] if x else 'No match')
        df['Score'] = df['Matched Template and Score'].apply(lambda x: x[1] if x else 0)
        df.drop('Matched Template and Score', axis=1, inplace=True)
        df = df.rename(columns={"Column 1": "Entity", "Column 2": "Votes"})
        # Align rows before any other operations
        df = align_rows(df, template)

        # Find indices for the first and last key template entries
        first_relevant_index = df[df['Matched Template'] == template[0]].index.min()
        last_relevant_index = df[df['Matched Template'] == template[-1]].index.max()

        # Apply conditions to filter out 'No match' entries where needed
        if pd.notna(first_relevant_index) and pd.notna(last_relevant_index):
            # Remove 'No match' before the first relevant entry
            df = df.drop(df[(df.index < first_relevant_index) & (df['Matched Template'] == 'No match')].index)
            # Remove any entries after the last relevant entry
            df = df.drop(df[df.index > last_relevant_index].index)
        # Rename columns and keep necessary ones
        df = df[['Entity', 'Matched Template', 'Score', 'Votes', 'confidence']]
    return df

In [5]:
# def get_total_votes_old(df_ocr):
#     try:
#         df_ocr['text'] = df_ocr['text'].str.strip().str.replace(r'\s+', '', regex=True)
#         # Calculate page height
#         page_height = df_ocr['y2'].max()

#         # Identify the largest table
#         tables = df_ocr[df_ocr['label'] == 'table']
#         if len(tables) < 2:
#             return -1
#         tables = df_ocr[df_ocr['label'] == 'table'].sort_values(by='y1')
#         largest_table = tables.iloc[0] if not tables.empty else None

#         # Calculate the bottom 1/3 of the page
#         bottom_third = page_height * 2 / 3

#         # Calculate the bottom 1/5 of the largest table
#         largest_table_bottom_fifth = largest_table['y2'] * 4 / 5

#         # Identify 'TOTAL VALID VOTES' in the required area
#         # target_text = 'TOTAL VALID VOTES'
#         # target_boxes = df_ocr[(df_ocr['y1'] > bottom_third) & (df_ocr['y1'] > largest_table_bottom_fifth) &
#         #                       (df_ocr.apply(lambda row: fuzz.token_sort_ratio(row['text'], target_text) > 85, axis=1))]
#         target_text = 'VALID VOTES'
#         df_ocr['match_ratio'] = df_ocr['text'].apply(lambda text: fuzz.token_sort_ratio(text, target_text))
#         target_boxes = df_ocr[(df_ocr['y1'] > bottom_third) & (df_ocr['y1'] > largest_table_bottom_fifth) &
#                               (df_ocr['match_ratio'] > 50)]

#         if target_boxes.empty:
#             return -1

#         # Use the box with the highest matching ratio
#         target_box = target_boxes.loc[target_boxes['match_ratio'].idxmax()]
#         box_height = target_box['y2'] - target_box['y1']
#         box_width = target_box['x2'] - target_box['x1']

#         # Define the search zone for integer values
#         ymin = int(target_box['y1'] - box_height)
#         ymax = int(target_box['y2'] + box_height)
#         xmin = int(target_box['x2'] - box_width/2)
#         xmax = int(2 * (largest_table['x2'] + largest_table['x1']) / 2)
#         search_box_coords = [xmin, ymin, xmax, ymax]
#         # Find all integers in the defined zone
#         potential_integers = df_ocr[(df_ocr['label'] == 'box') & (df_ocr.apply(lambda row: calculate_iou_inside(
#             [row['x1'], row['y1'], row['x2'], row['y2']], search_box_coords) > 0.5, axis=1))]
#         integer_values = []
#         for text in potential_integers['text']:
#             try:
#                 if text.isdigit():
#                     integer_values.append(int(text))
#             except:
#                 pass
#         if not integer_values:
#             return -1
#         # Return the largest integer found

#         return max(integer_values)
#     except:
#         return -1
def get_total_votes(df_ocr):
    def has_num(s):
        return any(i.isdigit() for i in s)
    def count_numbers(s):
        return sum(1 for i in s if i.isdigit())
    def clean(text):
        # Use regular expression to remove any character that is not a-z, A-Z, or 0-9
        cleaned_text = re.sub(r'[^a-zA-Z0-9]', '', text)
        return cleaned_text
    def contains_number_words(s):
        # List of number words to check
        number_words = ['one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'ten', 'hundred']
        # Convert the input string to lowercase
        s_lower = s.lower()
        # Check if any of the number words are in the string
        found_words = [word for word in number_words if word in s_lower]
        # Return the found number words or an empty list if none are found
        return found_words
    def max_integer_value(lst):
        integers = [item for item in lst if isinstance(item, int)]
        if integers:
            return max(integers)
        else:
            return -1
    try:
        df_ocr['text'] = df_ocr['text'].str.strip().str.replace(r'\s+', '', regex=True)
        # Calculate page height
        page_height = df_ocr['y2'].max()

        # Identify the largest table
        tables = df_ocr[df_ocr['label'] == 'table']
        if len(tables) < 2:
            return -1
        tables = df_ocr[df_ocr['label'] == 'table'].sort_values(by='y1', ascending = False)
        largest_table = tables.iloc[0] if not tables.empty else None
        # Calculate the bottom 1/3 of the page
        bottom_third = page_height * 2 / 3
        all_stamps = df_ocr[df_ocr['label'] == 'stamp']
        if len(all_stamps) < 2:
            return -1
        try:
            stamp_top = df_ocr[df_ocr['label'] == 'stamp'].sort_values(by='y1', ascending = False).iloc[0]['y1']
            stamp_top = max(stamp_top, bottom_third)
            stamp_top = min(stamp_top, page_height)
        except:
            stamp_top = page_height * .9
        # Calculate the bottom 1/5 of the largest table
        largest_table_bottom_fifth = largest_table['y2'] - (largest_table['y2'] - largest_table['y1']) / 5
        largest_table_lowerlimit = largest_table['y2'] + (largest_table['y2'] - largest_table['y1']) / 15
        target_text = 'TOTAL VALID VOTES'
        df_ocr['match_ratio'] = df_ocr['text'].apply(lambda text: fuzz.token_sort_ratio(text, target_text))
        target_boxes = df_ocr[(df_ocr['match_ratio'] > 50) & (df_ocr['y1'] > bottom_third) & (df_ocr['y1'] < stamp_top) & (df_ocr['y1'] > largest_table_bottom_fifth) & (df_ocr['y1'] < largest_table_lowerlimit)]
        if target_boxes.empty:
            ymax = largest_table['y2']
            ymin = int(largest_table['y2'] - (largest_table['y2'] - largest_table['y1']) / 5)
            xmin = int(largest_table['x1'] + (largest_table['x2'] - largest_table['x1']) / 5)
            xmax = int(largest_table['x1'] + (largest_table['x2'] - largest_table['x1']) * 3/ 5)
            search_box_coords = [xmin, ymin, xmax, ymax]

        else:
        # Use the box with the highest matching ratio
            target_box = target_boxes.loc[target_boxes['match_ratio'].idxmax()]
            box_height = target_box['y2'] - target_box['y1']
            box_width = target_box['x2'] - target_box['x1']
            # Define the search zone for integer values
            ymin = int(target_box['y1'] - box_height)
            ymax = min(stamp_top, int(target_box['y2'] + box_height))
            xmin = int(target_box['x2'] - box_width/2)
            xmax = int(2 * (largest_table['x2'] + largest_table['x1']) / 2)
            search_box_coords = [xmin, ymin, xmax, ymax]
        # Find all integers in the defined zone

        potential_integers = df_ocr[(df_ocr['label'] == 'box') & (df_ocr.apply(lambda row: calculate_iou_inside(
            [row['x1'], row['y1'], row['x2'], row['y2']], search_box_coords) > 0.5, axis=1))]
        integer_values = []
        for rawtext in potential_integers['text']:
            try:
                text = clean(rawtext)
                if len(text) <= 10 and (text.strip().isdigit() or text.strip()[0].isdigit() or count_numbers(text.strip()) >=2):
                    try:
                        integer_values.append(int(rawtext.strip()))
                        continue
                    except:
                        integer_values.append(str(rawtext))
                        pass
                elif contains_number_words(text):
                    integer_values.append(str(rawtext))
            except:
                continue
        if not integer_values:
            return -1
        try:
            item = max_integer_value(integer_values)
            if item!= -1:
                return item
            else:
                return integer_values + ['<check>']
        except:
            return integer_values + ['<check>']
    except:
        return -1

In [6]:
def create_key_value_pair(df_ocr):
    data = df_ocr
    kvs = data[data['label'] == 'kv'].copy().sort_values(by='y1')
    texts = []
    for i in range(len(kvs)):
        if not pd.isna(kvs.iloc[i]['text']):
          texts.append(kvs.iloc[i]['text'])
    # Check if there are exactly four entries with the word 'Code'
    code_texts = [text for text in texts if 'Code' in text]
    if len(code_texts) == 4:
        # Extract the numeric parts
        code_values = [int(''.join(filter(str.isdigit, text.replace('o', '0').replace('O', '0')))) for text in code_texts]

        # Map to the required dictionary
        try:
            if code_value[0] /1000 > 1:
                code_values[0] = -1
            if code_value[1] /100 > 1:
                code_values[1] = -1
            if code_value[2] /100 > 1:
                code_values[2] = -1
            if code_value[3] /1000 > 1:
                code_values[3] = -1
        except:
            pass
        code_dict = {
            'STATE CODE': code_values[0],
            'LGA CODE': code_values[1],
            'RA CODE': code_values[2],
            'PU CODE': code_values[3]
        }

        return code_dict
    else:
        code_dict = {
            'STATE CODE': -1,
            'LGA CODE': -1,
            'RA CODE': -1,
            'PU CODE': -1
        }
        return code_dict

In [7]:
def inside_iou_custom(box1, box2):
    x1_inter = max(box1['x1'], box2['x1'])
    y1_inter = max(box1['y1'], box2['y1'])
    x2_inter = min(box1['x2'], box2['x2'])
    y2_inter = min(box1['y2'], box2['y2'])
    inter_area = max(0, x2_inter - x1_inter) * max(0, y2_inter - y1_inter)
    box1_area = (box1['x2'] - box1['x1']) * (box1['y2'] - box1['y1'])
    box2_area = (box2['x2'] - box2['x1']) * (box2['y2'] - box2['y1'])
    if min(box1_area, box2_area) == 0:
        return 0
    return inter_area / min(box1_area, box2_area)

# Function to get the top table from the dataframe
def get_top_table(df):
    # Filter the dataframe to get only table-related entries
    table_entries = df[df['label'] == 'table']
    unique_tables = table_entries[['x1', 'y1', 'x2', 'y2']].drop_duplicates()

    top_table_entry = unique_tables.sort_values('y1').iloc[0]
    return top_table_entry

# Function to get the right column inside the top table
def get_right_column(df, table):
    # Extract the bounding box of the top table
    table_box = {'x1': table['x1'], 'y1': table['y1'], 'x2': table['x2'], 'y2': table['y2']}

    # Filter entries for columns within the top table's bounding box
    column_entries = df[df['label'] == 'column']

    # Filter columns based on inside_iou with the table
    columns_inside = []
    for _, col in column_entries.iterrows():
        col_box = {'x1': col['x1'], 'y1': col['y1'], 'x2': col['x2'], 'y2': col['y2']}
        iou = inside_iou_custom(table_box, col_box)
        if iou > .5:  # Check if the column is inside the table
            columns_inside.append(col)

    # Convert to dataframe and sort by x1
    if columns_inside:
        columns_inside_df = pd.DataFrame(columns_inside).sort_values('x1')
        # Return the column with the maximum x1 (right-most column)
        return columns_inside_df.iloc[-1]
    else:
        return None

def get_box_inside_column(df, column):
    # Extract bounding box of the column
    column_box = {'x1': column['x1'], 'y1': column['y1'], 'x2': column['x2'], 'y2': column['y2']}

    # Filter for 'box' entries
    box_entries = df[df['label'] == 'box']

    # Check for each box if it intersects with the filtered column
    intersecting_boxes = []
    for _, box in box_entries.iterrows():
        iou = inside_iou_custom(column_box, box)
        if iou > .2:
            intersecting_boxes.append(box)

    return pd.DataFrame(intersecting_boxes)

# Define the complete pipeline function
def top_table_pipeline(df):
    # Get top table
    top_table = get_top_table(df)

    # Get right column of the top table
    right_column = get_right_column(df, top_table)
    if right_column is not None:
        # Get boxes inside the right column
        boxes_inside_column = get_box_inside_column(df, right_column)
        # Return the dataframe with boxes inside column
        return boxes_inside_column, right_column
    else:
        return None
def get_top_bottom_numbers(df, table):
    # Extract table bounding box coordinates
    table_y1, table_y2 = table['y1'], table['y2']

    # Calculate the height of the table
    table_height = table_y2 - table_y1

    # Define the top and bottom 25% thresholds
    top_25_threshold = table_y1 + 0.30 * table_height
    bottom_25_threshold = table_y2 - 0.30 * table_height
    # Filter for entries within the top and bottom 25% of the table
    top_25_df = df[(df['y2'] >= table_y1) & (df['y1'] <= top_25_threshold)]
    bottom_25_df = df[(df['y2'] >= bottom_25_threshold) & (df['y1'] <= table_y2)]
    # Calculate midline for both top and bottom 25% sections
    top_midline = (table_y1 + top_25_threshold) / 2
    bottom_midline = (bottom_25_threshold + table_y2) / 2
    # Extract numbers from top and bottom 25% sections
    def get_number_from_df(df_section, midline):
        # Calculate the centroid of each cell (average of y1 and y2)
        df_section['centroid_y'] = (df_section['y1'] + df_section['y2']) / 2

        # Filter for rows above the midline
        above_midline = df_section[df_section['centroid_y'] <= midline]

        # Get the closest number above the midline, prioritizing integers
        if not above_midline.empty:
            above_midline['distance_to_midline'] = abs(above_midline['centroid_y'] - midline)

            # Prioritize integers
            integer_above = above_midline[above_midline['text'].apply(lambda x: str(x).isdigit())]
            if not integer_above.empty:
                number_above = integer_above.loc[integer_above['distance_to_midline'].idxmin(), 'text']
            else:
                number_above = above_midline.loc[above_midline['distance_to_midline'].idxmin(), 'text']
        else:
            number_above = None

        # Filter for rows below the midline
        below_midline = df_section[df_section['centroid_y'] > midline]

        # Get the closest number below the midline, prioritizing integers
        if not below_midline.empty:
            below_midline['distance_to_midline'] = abs(below_midline['centroid_y'] - midline)

            # Prioritize integers
            integer_below = below_midline[below_midline['text'].apply(lambda x: str(x).isdigit())]
            if not integer_below.empty:
                number_below = integer_below.loc[integer_below['distance_to_midline'].idxmin(), 'text']
            else:
                number_below = below_midline.loc[below_midline['distance_to_midline'].idxmin(), 'text']
        else:
            number_below = None

        return number_above, number_below

    # Get numbers from top and bottom 25% of the table
    number1, number2 = get_number_from_df(top_25_df, top_midline)
    number3, number4 = get_number_from_df(bottom_25_df, bottom_midline)
    # Return the numbers in JSON format
    return pd.DataFrame({
        "Number of Voters on the Register": number1,
        "Number of Accredited Voters": number2,
        "Number of Rejected Ballot": number3,
        "Total Number at used Ballot Papers": number4
    }.items())



## Test Function with a single image

In [8]:
# Test function to generate CSV with layout model + Amazon boto3
# image_path='../test_images/batch_test/unwrapped_files_jpg/state_01/lga_01/ward_03/01_01_03_004_unwrap.jpg'
# df_textract_test = extract_textract(image_path)
# df_yolo_test = get_yolo_boxes(image_path)
# df_ocr_test = match_and_draw(image_path, df_textract_test,df_yolo_test,'../csv/ocr/' , mode='process', save_csv=True)

NameError: name 'image_path' is not defined

In [ ]:
df_ocr_test

In [ ]:
top_table_test = match_top_table(create_top_table(df_ocr_test))
bottom_table_test = postprocess_bottom_table(create_bottom_table(df_ocr_test))
key_value_pair_test = create_key_value_pair(df_ocr_test)

In [ ]:
bottom_table_test

## Batch run extraction function on all papers

In [ ]:
# ## get the image path which has not run ocr and layout extraction
# import pandas as pd
# df_random=pd.read_csv('../../shared_data/csv/ocr/files_to_run_mark.csv')
# selected_paths = df_random['unwrap_path2'].sample(n=min(2000, len(df_random)), random_state=25).tolist()
# # Create a mask for paths not in the previous sample
# mask = ~df_random['unwrap_path2'].isin(selected_paths)

# # Create a new DataFrame excluding the previously selected paths
# df_new = df_random[mask]

# all_paths=df_new['unwrap_path2'].tolist()

# all_paths[15083]

In [9]:
df_files=pd.read_csv('../test_images/batch_test/csv/images_unwrap_process.csv')

In [10]:
# Load the image-processing metadata and keep only rows that successfully completed the earlier preprocessing step.
# The new unwrap_path column points to the normalized image version used in the layout workflow.
csv_file = '../test_images/batch_test/csv/images_unwrap_process.csv'
df_files=pd.read_csv(csv_file)
def update_file_path(file_path):
    # Converts the original cropped-image path into the corresponding unwrapped-image path.
    # Application purpose: links each metadata row to the preprocessed image version used for layout analysis.
    p1=file_path.replace('consolidated_files_jpg', '../test_images/batch_test/unwrapped_files_jpg')
    p2=p1.replace('.jpg', '_unwrap.jpg')
    return p2

df_files['unwrap_filepath'] = df_files['jpg_filepath'].apply(update_file_path)
df_files.head()

,jpg_filepath,polling_unit_code,blurred,status,unwrap_filepath
0,consolidated_files_jpg/state_22/lga_05/ward_10...,22/05/10/028,False,Success,../test_images/batch_test/unwrapped_files_jpg/...
1,consolidated_files_jpg/state_22/lga_05/ward_10...,22/05/10/016,False,Success,../test_images/batch_test/unwrapped_files_jpg/...
2,consolidated_files_jpg/state_22/lga_05/ward_10...,22/05/10/047,False,Success,../test_images/batch_test/unwrapped_files_jpg/...
3,consolidated_files_jpg/state_22/lga_05/ward_10...,22/05/10/010,False,Success,../test_images/batch_test/unwrapped_files_jpg/...
4,consolidated_files_jpg/state_02/lga_14/ward_04...,02/14/04/023,False,Success,../test_images/batch_test/unwrapped_files_jpg/...


In [11]:
all_paths=df_files['unwrap_filepath'].tolist()


In [12]:
def check_file_exists(file_path):
    return os.path.exists(file_path)

In [ ]:
# ## batch run extraction on images which hasn't, save the OCR layout csv to a folder, save the failed images in a txt file
# failed_paths_file = '../csv/ocr/summary/failed_paths.txt'

# try:
#     with open(failed_paths_file, 'r') as f:
#         failed_paths = f.read().splitlines()
# except FileNotFoundError:
#     failed_paths = []


# def save_failed_paths(failed_paths, output_file):
#     with open(output_file, 'w') as f:
#         for path in failed_paths:
#             f.write(f"{path}\n")


# for image_path in tqdm(all_paths):
#     if check_file_exists(image_path)==False:
#       print(image_path+ " path not exist")
#       failed_paths.append(image_path)
#       save_failed_paths(failed_paths, failed_paths_file)
#     try:
#       df_textract = extract_textract(image_path)
#       df_yolo = get_yolo_boxes(image_path)
#       df_ocr = match_and_draw(image_path, df_textract,df_yolo,'../csv/ocr/all_df_ocr/' , mode='process', save_csv=True)
#       print(all_paths.index(image_path))
#     except:
#       failed_paths.append(image_path)
#       save_failed_paths(failed_paths, failed_paths_file)
#       print(image_path)
#       pass



In [18]:
from tqdm import tqdm
import os
failed_path=[]

for image_path in tqdm(all_paths):
    # df_textract = extract_textract(image_path)
    # df_yolo = get_yolo_boxes(image_path)
    if check_file_exists(image_path)==False:
      print(image_path+ " path not exist")
    try:
        # df_ocr = match_and_draw(image_path, df_textract,df_yolo,'../../shared_data/csv/ocr/all_csv_df_ocr/' , mode='process', save_csv=True)
        df_ocr = pd.read_csv('../csv/ocr/all_df_ocr/'+os.path.basename(image_path).replace('.jpg','.csv'))
        try:
            top_table = match_top_table(create_top_table(df_ocr))
            top_table['image_path'] = image_path
            top_table.to_csv('../csv/ocr/summary/top_table.csv', index=False, mode='a', header=not check_file_exists('../csv/ocr/summary/top_table.csv'))
        except Exception as e:
            print(f"Error processing top_table for {image_path}: {e}")
            pass

        try:
            bottom_table = postprocess_bottom_table(create_bottom_table(df_ocr))
            bottom_table['image_path'] = image_path
            bottom_table.to_csv('../csv/ocr/summary/bottom_table.csv', index=False, mode='a', header=not check_file_exists('../csv/ocr/summary/bottom_table.csv'))
        except Exception as e:
            print(f"Error processing bottom_table for {image_path}: {e}")
            pass

        try:
            key_value_pair = create_key_value_pair(df_ocr)
            key_value_pair['image_path'] = image_path
            key_value_pair_df = pd.DataFrame([key_value_pair])
            key_value_pair_df.to_csv('../csv/ocr/summary/key_value_pair.csv', index=False, mode='a', header=not check_file_exists('../csv/ocr/summary/key_value_pair.csv'))
        except Exception as e:
            print(f"Error processing key_value_pair for {image_path}: {e}")
            pass
        try:
            vote_list = []
            tot_votes = get_total_votes(df_ocr)
            vote_list.append([image_path, tot_votes])
            total_vote_df = pd.DataFrame(vote_list, columns = ['image_path', 'total_votes'])
            total_vote_df.to_csv('../csv/ocr/summary/total_votes.csv', index=False, mode='a', header=not check_file_exists('../csv/ocr/summary/total_votes.csv'))
        except:
            pass

        ## revised function for top table extraction
        try:
            result_df, right_column = top_table_pipeline(df_ocr)
            top_table_numbers = get_top_bottom_numbers(result_df, right_column)
            top_table_numbers['image_path'] = image_path
            top_table_numbers.to_csv('../csv/ocr/summary/top_table_numbers.csv', index=False, mode='a', header=not check_file_exists('../csv/ocr/summary/top_table_numbers.csv'))
        except Exception as e:
            print(f"Error processing top_table_number for {image_path}: {e}")
            df_ocr = pd.DataFrame()
            pass
    except:
        # pass
        failed_path.append(image_path)
        print(image_path)
        pass


 48%|█████████████████████████████████████████████                                                 | 12/25 [00:01<00:00, 13.21it/s]

../test_images/batch_test/unwrapped_files_jpg/state_01/lga_02/ward_03/01_02_03_001_unwrap.jpg path not exist
../test_images/batch_test/unwrapped_files_jpg/state_01/lga_02/ward_03/01_02_03_001_unwrap.jpg
../test_images/batch_test/unwrapped_files_jpg/state_22/lga_05/ward_10/22_05_10_019_unwrap.jpg path not exist
../test_images/batch_test/unwrapped_files_jpg/state_22/lga_05/ward_10/22_05_10_019_unwrap.jpg
../test_images/batch_test/unwrapped_files_jpg/state_22/lga_03/ward_02/22_03_02_008_unwrap.jpg path not exist
../test_images/batch_test/unwrapped_files_jpg/state_22/lga_03/ward_02/22_03_02_008_unwrap.jpg
Error processing top_table for ../test_images/batch_test/unwrapped_files_jpg/state_01/lga_01/ward_01/01_01_01_015_unwrap.jpg: expected string or bytes-like object


 56%|████████████████████████████████████████████████████▋                                         | 14/25 [00:01<00:01, 10.39it/s]

Error processing bottom_table for ../test_images/batch_test/unwrapped_files_jpg/state_01/lga_01/ward_01/01_01_01_015_unwrap.jpg: 'Political Party'
Error processing top_table for ../test_images/batch_test/unwrapped_files_jpg/state_01/lga_01/ward_01/01_01_01_003_unwrap.jpg: expected string or bytes-like object


 64%|████████████████████████████████████████████████████████████▏                                 | 16/25 [00:01<00:00, 11.70it/s]

Error processing bottom_table for ../test_images/batch_test/unwrapped_files_jpg/state_01/lga_01/ward_01/01_01_01_003_unwrap.jpg: 'NoneType' object has no attribute 'columns'
Error processing top_table for ../test_images/batch_test/unwrapped_files_jpg/state_01/lga_01/ward_01/01_01_01_002_unwrap.jpg: 'NoneType' object does not support item assignment
Error processing bottom_table for ../test_images/batch_test/unwrapped_files_jpg/state_01/lga_01/ward_01/01_01_01_002_unwrap.jpg: 'NoneType' object has no attribute 'columns'
Error processing top_table_number for ../test_images/batch_test/unwrapped_files_jpg/state_01/lga_01/ward_01/01_01_01_002_unwrap.jpg: single positional indexer is out-of-bounds


 80%|███████████████████████████████████████████████████████████████████████████▏                  | 20/25 [00:02<00:00, 10.50it/s]

Error processing top_table for ../test_images/batch_test/unwrapped_files_jpg/state_01/lga_04/ward_03/01_04_03_017_unwrap.jpg: 'NoneType' object does not support item assignment
Error processing bottom_table for ../test_images/batch_test/unwrapped_files_jpg/state_01/lga_04/ward_03/01_04_03_017_unwrap.jpg: 'NoneType' object has no attribute 'columns'
Error processing top_table_number for ../test_images/batch_test/unwrapped_files_jpg/state_01/lga_04/ward_03/01_04_03_017_unwrap.jpg: single positional indexer is out-of-bounds


100%|██████████████████████████████████████████████████████████████████████████████████████████████| 25/25 [00:03<00:00,  8.30it/s]


In [19]:
df_result=pd.read_csv('../csv/ocr/summary/bottom_table.csv')